#### 1
Crato notebook che useremo per vedere i dataset, tipo di dati forma , quanti NaN, distribuzione ecc

In [ ]:
import sys
sys.path.append("..")
import csv
from pathlib import Path

Leggiamo la prima riga del file x_train per capire cosa abbiamo
Capita la dimensione della tabella, contiamo il numero di NaN per feature:

In [ ]:
# Percorso al file x_train
file_path_x = Path('../data/x_train.csv')

with open(file_path_x, mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)
    
    # 1. Read header
    headers = next(reader)
    num_cols = len(headers)

    print(f"Number of coloumns : {num_cols}")
    print(f"Header: {headers}\n")
    
    # 2. Contiamo i NaN riga per riga in modo efficiente
    total_rows = 0

    # create a vector where in each cell we put the number of NaN in that coloumn
    nan_counts = [0] * num_cols 

    # interate for each row
    for row in reader:
        total_rows += 1
        # uso enumerate perchè così posso prendere sia la posizione col_index che il valore value, tipo un dizionario
        for col_index, value in enumerate(row):
            # if the cell is empty or 'NaN', increase the counter in that cell
            if value == '' or value.lower() == 'nan':
                nan_counts[col_index] += 1

    print(f"Number of rows : {total_rows}")

#### 2 
Adesso importiamo il file y_train e contiamo quanti sono i valori di 1 e di -1, per capire quanto sono sbilanciate le due classi

In [ ]:
file_path_y = Path('../data/y_train.csv')

# Dizionario standard con contatori già pronti
dict = {-1: 0, 1: 0}

with open(file_path_y, mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)
    
    # 1. Leggiamo l'header
    headers_y = next(reader)

    # iteriamo sulle colonne prendendo solo la 2 che è quella che ha le nostre lable:
    for row in reader:
        valore_str = row[1]

        valore = int(valore_str) # converto a intero
        
        # Incrementiamo il contatore corrispondente nel dizionario
        if valore in dict:
                dict[valore] += 1

for valore, conteggio in dict.items():
    print(f"Valore {valore}: {conteggio} volte")

#### 3
Ora che sappiamo il numero di NaN per ogni colonna, conviene contare quante sono le colonne che hanno un numero di NaN maggiore di soglia_NaN

In [ ]:
# 1. Creiamo una lista vuota per raccogliere i dati delle colonne con NaN
colonne_con_nan = []

for i, col_name in enumerate(headers):
    if nan_counts[i] > 0:
        perc = (nan_counts[i] / total_rows) * 100
        # Salviamo un pacchetto (nome_colonna, numero_nan, percentuale)
        colonne_con_nan.append((col_name, nan_counts[i], perc))

# 2. Ordiniamo la lista in ordine DECRESCENTE basandoci sulla percentuale (che è in posizione 2)
# reverse=True serve proprio a partire dal valore più alto
colonne_con_nan.sort(key=lambda x: x[2], reverse=True)

# 3. Stampiamo il risultato ordinato e contiamo quelle > 50%
useless_coloumns = 0
candidate_da_testare = []

# parametro che mi dice quale soglia di NaN tenere o meno
soglia_NaN = 50

print("Colonne ordinate per % di NaN (dal più alto al più basso):\n")
for col_name, count, perc in colonne_con_nan:
    if perc > soglia_NaN:
        useless_coloumns += 1
        candidate_da_testare.append(col_name)

print(f"\nPossibili colonne da scartare (> {soglia_NaN}%): {useless_coloumns} su {len(headers)}")
print(candidate_da_testare)

## Iniziamo a togliere alcuni valori dalla tabella

Dato che 322 valori sono troppi e molte sono ingormazioni inutili, possiamo scartare colonne come:
Id, _STATE, FMONTH, IDATE, IMONTH, IDAY, IYEAR, STATERES

Amministrative (come è stata gestita la chiamata)
DISPCODE, SEQNO, _PSU, CTELENUM, PVTRESD1, CELLFON3, NUMHHOL2, CPDEMO1, QSTVER, QSTLANG, MSCODE

Pesi statistici del sondaggio (servono ai ricercatori per ricalibrare il campione, non descrivono la persona)
_STSTR, _STRWT, _RAWRAKE, _WT2RAKE, _DUALUSE, _DUALCOR, _LLCPWT

In [ ]:
# lista che mi contiene quale colonne togliere dal mio dataset perchè contengono dati irrilevanti: 

da_togliere = []

da_togliere = [
    # Identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    
    # Amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3', 
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    
    # Pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT'
]

print(da_togliere)

#### Rilevanza colonne con tanti NaN
Andiamo a verificare quante delle possibili colonne che si potrebbero scartare dato il gran numero di NaN sono rilevanti o meno

In [ ]:
# Carichiamo y train: 
y_path = Path('../data/y_train.csv')
y_labels = []

# leggiamo tutti i valori di +1 e -1 dal mio dataset y
with open(y_path, mode='r', encoding='utf-8') as f:
    reader_y = csv.reader(f)
    next(reader_y)  # salta header
    for row in reader_y:
        if row[1] != '':
            y_labels.append(int(row[1]))

# Calcoliamo la baseline globale di y = 1
totale_y = len(y_labels)
totale_uni = y_labels.count(1)
baseline_perc = (totale_uni / totale_y) * 100
print(f" Baseline globale di y = 1: {baseline_perc:.2f}%\n") 

# --- PASSO 2: Prepariamo le colonne candidate da testare ---

# Dizionari per contare i dati per ogni colonna candidata
# es. valid_counts['COLGHOUS'] = quante persone hanno risposto
valid_counts = {col: 0 for col in candidate_da_testare}
ones_counts = {col: 0 for col in candidate_da_testare}

# --- PASSO 3: Leggiamo x_train riga per riga con il 'with' ---
x_path = Path('../data/x_train.csv')

with open(x_path, mode='r', encoding='utf-8') as f:
    reader_x = csv.reader(f)
    
    # Leggiamo l'header per mappare i nomi alle posizioni (indici)
    headers = next(reader_x)
    
    # Troviamo gli indici delle colonne candidate presenti nell'header
    # Creiamo un dizionario {indice_colonna: nome_colonna} solo per le nostre candidate
    indici_da_cercare = {}
    for idx, col_name in enumerate(headers):
        if col_name in candidate_da_testare:
            indici_da_cercare[idx] = col_name

    # Scorriamo riga per riga associandole alla y corrispondente tramite un indice numerico (row_idx)
    for row_idx, row in enumerate(reader_x):
        # Prendiamo la y di questa persona
        current_y = y_labels[row_idx]
        
        # Controlliamo solo le colonne candidate che ci interessano in questa riga
        for col_idx, col_name in indici_da_cercare.items():
            valore = row[col_idx]
            
            # Se il valore NON è vuoto e NON è NaN
            if valore != '' and valore.lower() != 'nan':
                valid_counts[col_name] += 1
                if current_y == 1:
                    ones_counts[col_name] += 1

# --- PASSO 4: Calcoliamo i risultati e ordiniamo per scostamento ---
risultati = []

for col in candidate_da_testare:
    n_validi = valid_counts[col]
    
    # Se troppi pochi dati (es. meno di 50 persone hanno risposto), saltiamo per sicurezza
    if n_validi < 50:
        continue
        
    perc_gruppo = (ones_counts[col] / n_validi) * 100
    scostamento = abs(perc_gruppo - baseline_perc)
    
    risultati.append((col, n_validi, perc_gruppo, scostamento))

# Ordiniamo dal più "anomalo" (distante dall'8.8%) al più normale
risultati.sort(key=lambda x: x[3], reverse=True)

# Stampiamo le prime 20 colonne più interessanti (quelle da SALVARE nonostante i NaN)
print("\n" + "="*75)
print(f"{'Colonna':<15} | {'Valori validi':<14} | {'% y=1 nel gruppo':<18} | {'Scostamento':<12}")
print("="*75)

for col, n_val, perc, diff in risultati[:20]:
    print(f"{col:<15} | {n_val:<14} | {perc:6.2f}%            | {diff:6.2f}%")


## Standardizzazione

Troppa roba da vedere con colonne uguali quindi ignoriamo, teniamo tutto e poi ottimizziamo


In [ ]:
import numpy as np
from pathlib import Path
from helpers import load_csv_data

data_path = Path('../data')

# 1. Header: li leggo a parte, perché load_csv_data non li restituisce
with open(data_path / 'x_train.csv', mode='r', encoding='utf-8') as f:
    headers = f.readline().strip().split(',')
headers = headers[1:]  # tolgo 'Id', che load_csv_data mette in train_ids

# 2. Dati numerici (le celle vuote diventano NaN)
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
print("x_train:", x_train.shape, "| header:", len(headers))  # le colonne devono coincidere

# 3. Tolgo le colonne tecniche (Id è già fuori)
da_togliere = {
    '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3',
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT',
}
keep = [i for i, name in enumerate(headers) if name not in da_togliere]
x = x_train[:, keep]
names = [headers[i] for i in keep]
print("Colonne rimaste:", x.shape[1])

# 4. Valori distinti per colonna (ignorando i NaN)
n_distinti = np.zeros(x.shape[1], dtype=int)
for j in range(x.shape[1]):
    col = x[:, j]
    n_distinti[j] = len(np.unique(col[~np.isnan(col)]))

# 5. Conteggio con soglia 15
soglia = 15
poche = np.where(n_distinti < soglia)[0]
molte = np.where(n_distinti >= soglia)[0]
print(f"Meno di {soglia} valori distinti: {len(poche)}")
print(f"{soglia} o più valori distinti:  {len(molte)}")

# 6. Qualche nome per controllare a occhio
print("\nEsempi con pochi valori:", [names[j] for j in poche[:15]])
print("Esempi con molti valori:", [names[j] for j in molte[:15]])
print("Colonne costanti (<= 1 valore):", [names[j] for j in np.where(n_distinti <= 1)[0]])